# Baseline Churn Model Check + Initial Behavioral Drift Score

Input: `data/features_and_label.parquet` (customer-month rows from notebook 03, label `is_churn`)

Output: drift features + `drift_score` per customer-month

1. Verify that the baseline churn models from 06 beat trivial baselines and are not fooled by leakage
2. Build the first behavioral-drift score & test whether it carries churn signal beyond th existing features


## 1. Setup + Load Data

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

In [4]:
DATA_PATH     = Path('../data/features_and_label.parquet')
OUT_PATH      = Path('../data/drift_scores.parquet')
TARGET        = 'is_churn'

TEST_FRACTION = 0.20   # same as notebook 06: latest 20% of distinct months are the test set

PURGE_MONTHS  = 3      # = the 90-day label window (~3 months)

K_BASELINE    = 3      # a customer's "normal" = mean of their previous 3 ACTIVE months
CLIP_Z        = 3.0    # cap each standardized drift component at +/-3 so one freak month can't dominate
RANDOM_STATE  = 42

In [5]:
df = pd.read_parquet(DATA_PATH)

# CustomerMonth is stored as a Period[M], so we'll convert to a month-start timestamp (same handling as notebook 06)
cm = df['CustomerMonth']
df['CustomerMonth'] = cm.dt.to_timestamp() if isinstance(cm.dtype, pd.PeriodDtype) else pd.to_datetime(cm)

df = df.dropna(subset=[TARGET, 'CustomerMonth']).copy()
df[TARGET] = df[TARGET].astype(int)
df['is_first_month'] = df['is_first_month'].astype(int)   # bool -> 0/1 so every model treats it as a plain number

# sort by customer then month
# all "previous months" logic below (shift/rolling) depends on this order
df = df.sort_values(['Customer ID', 'CustomerMonth']).reset_index(drop=True)

print(f'Rows: {len(df):,} | customers: {df["Customer ID"].nunique():,} | churn rate: {df[TARGET].mean():.2%}')
print(f'Months: {df.CustomerMonth.min():%Y-%m} to {df.CustomerMonth.max():%Y-%m}')
display(df.head())

Rows: 20,880 | customers: 5,248 | churn rate: 38.99%
Months: 2009-12 to 2011-09


,Customer ID,CustomerMonth,purchase_frequency,total_spend,avg_order_value,unique_items,days_since_last_purchase,is_first_month,is_churn
0,12346,2010-03-01,1,27.05,27.05,5,NaN,1,1
1,12346,2010-06-01,1,142.31,142.31,19,118.0,0,1
2,12346,2011-01-01,1,149.76,149.76,1,203.0,0,1
3,12347,2010-10-01,1,611.53,611.53,40,NaN,1,0
4,12347,2010-12-01,1,711.79,711.79,31,37.0,0,0


## 2. Chronological Split with a Purge Gap

Same idea as notebook 06: train on earlier months & test on the latest 20% of distinct months. A random split would be wrong here, because it lets the model learn from the future.

- Added a purge gap of 3 months at the end of train

A training row from month *M* is labeled using purchases up to ~90 days after its last purchase, which reaches into months *M+1 ... M+3*. 

If the test period starts right after train, the final training labels are partly built from purchases that fall *inside the test period*, so the model has indirectly seen test-period behavior. 

Dropping the last `PURGE_MONTHS` months of train removes that overlap.

The cost is a smaller training set; the benefit is a test score that reflects what you'd get in production.


In [6]:
def make_split(data, purge_months):
    months = np.sort(data['CustomerMonth'].unique())
    cutoff_idx = min(max(1, int(np.floor(len(months) * (1 - TEST_FRACTION)))), len(months) - 1)
    cutoff = pd.Timestamp(months[cutoff_idx])
    train_end = cutoff - pd.DateOffset(months=purge_months)   # train uses months strictly before this
    train = data[data['CustomerMonth'] < train_end]
    test  = data[data['CustomerMonth'] >= cutoff]
    return train, test, cutoff, train_end

train_df, test_df, cutoff, train_end = make_split(df, PURGE_MONTHS)
train_nopurge, _, _, _ = make_split(df, 0)

print(f'Test starts:               {cutoff:%Y-%m}')
print(f'Purged train ends before:  {train_end:%Y-%m}  -> {len(train_df):,} rows, churn {train_df[TARGET].mean():.2%}')
print(f'Un-purged train (as in 06): {len(train_nopurge):,} rows')
print(f'Test:                      {len(test_df):,} rows, churn {test_df[TARGET].mean():.2%}')
assert train_df['CustomerMonth'].max() < test_df['CustomerMonth'].min(), 'train must precede test'

Test starts:               2011-05
Purged train ends before:  2011-02  -> 14,103 rows, churn 40.19%
Un-purged train (as in 06): 16,686 rows
Test:                      4,194 rows, churn 35.24%


## 3. Checking Baseline Models Work?

Using same three models and hyperparameters as notebook 06 (logistic regression, random forest, XGBoost), so results are comparable.

Headline metrics are ROC AUC, PR AUC, and lift in the top 10%

All three are threshold-free.

The models use `class_weight='balanced'` / `scale_pos_weight`, which shifts predicted probabilities so that the default 0.5 cutoff is arbitrary, and precision/recall/F1 at 0.5 mostly reflect that choice rather than model quality. 

Lift at 10% answers the business question "if we contact the 10% of customers the model ranks riskiest, how many more churners do we catch than by picking at random?"

In [7]:
BASE_FEATURES = ['purchase_frequency', 'total_spend', 'avg_order_value',
                 'unique_items', 'days_since_last_purchase', 'is_first_month']

def lift_at(y, score, frac=0.10):
    # churn rate among the top `frac` highest-scored rows, divided by the overall churn rate
    # Ties (e.g. a constant dummy score, or the many neutral-0 drift rows) are broken randomly with a fixed seed
    # Without this, tied rows would be ranked by row order and a constant score could show a fake lift
    y = np.asarray(y); score = np.asarray(score); k = max(1, int(len(y) * frac))
    order = np.random.default_rng(RANDOM_STATE).permutation(len(y))
    top = order[np.argsort(-score[order], kind='stable')[:k]]
    return y[top].mean() / y.mean()

def score_metrics(y, s):
    return {'ROC AUC': roc_auc_score(y, s), 'PR AUC': average_precision_score(y, s), 'Lift@10%': lift_at(y, s)}

def build_models(y_train):
    # same hyperparameters from notebook 06 on purpose (comparability)
    # Median imputation is only
    # needed for days_since_last_purchase in first-month rows; 
    # is_first_month tells the model which rows were imputed.
    return {
        'Logistic regression': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
            LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
        'Random forest': make_pipeline(SimpleImputer(strategy='median'),
            RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        'XGBoost': make_pipeline(SimpleImputer(strategy='median'),
            XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                          scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
                          eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)),
    }

def fit_and_eval(tr, te, features):
    models = build_models(tr[TARGET])
    rows, fitted = {}, {}
    for name, m in models.items():
        m.fit(tr[features], tr[TARGET])
        p_te = m.predict_proba(te[features])[:, 1]
        p_tr = m.predict_proba(tr[features])[:, 1]
        rows[name] = {**score_metrics(te[TARGET], p_te), 'Train ROC AUC': roc_auc_score(tr[TARGET], p_tr)}
        fitted[name] = m
    return pd.DataFrame(rows).T, fitted


## results
results_purged, fitted_base = fit_and_eval(train_df, test_df, BASE_FEATURES)
print(f'Test churn rate (= PR AUC of a random model): {test_df[TARGET].mean():.3f}')
display(results_purged.round(3))

Test churn rate (= PR AUC of a random model): 0.352


,ROC AUC,PR AUC,Lift@10%,Train ROC AUC
Logistic regression,0.766,0.626,2.066,0.734
Random forest,0.713,0.533,1.734,1.000
XGBoost,0.765,0.604,2.025,0.804
